## NYC 311 Data Ingestion Strategy 

### Objective

Retrieve the latest 90 days of NYC 311 complaint data from the Socrata API and persist the raw dataset in Amazon S3 while ensuring reliable pagination, fault tracking, and reproducible ingestion.

Data partitioned into 10-day chunks (90-day window) to reduce API load and improve reliability
Each chunk retrieved in 20K record batches using unique_key-based pagination to ensure ordering and avoid duplicates
Query design enforces deterministic sorting and controlled request sizes, mitigating API rate limits and timeouts

### Future Improvements

**Stream-based processing (fetch → transform → store) :** Incremental ingestion using stored metadata (last_created_date and last_unique_key) to retrieve only newly created records.


#### 1.1 Libarries Setup

In [4]:
from sodapy import Socrata
import pandas as pd

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

import time
import json
import os

import boto3
from io import BytesIO

#### 1.2 Function for Metadata Management

After a successful ingestion run, pipeline metadata is updated.

The metadata captures:

- latest processed timestamp
- latest processed unique_key
- ingestion timestamp

This information provides a checkpoint that can later be used to implement incremental ingestion instead of repeatedly performing full loads.

In [5]:
def update_meta_data(last_created_date,last_key):
    with open("current_state.json", "r") as f:
        state = json.load(f)

    print("LAST UPLOAD METADATA : ")
    print(state)

    state["last_created_date"] = last_created_date.isoformat()
    state["last_unique_key"] = int(last_key)
    state["last_ingest_time"] = datetime.now().isoformat()

    try:
        json.dumps(state)
        
        with open("current_state.json", "w") as f:
            json.dump(state, f, indent=4)

    except:
        print("Error")
    
    print("NEW UPLOAD METADATA : ")
    print(state)

    return

#### 1.3 Function To Maintain Load History
Each pipeline execution is logged separately.

For every run, the following information is recorded:

- execution time
- load type
- extraction window
- rows ingested
- files generated
- execution duration
- final processed key
- pipeline status

Maintaining execution history enables operational monitoring and simplifies troubleshooting.

In [6]:
def update_log_history(run_time,start_date,end_date,rows_loaded,files_created,duration_seconds,last_key):
    with open("load_history.json", "r") as f:
        history = json.load(f)

    print(history[-1])

    new_record =  {
        "run_time": run_time.isoformat(),
        "load_type": "FULL",
        "start_date": start_date.isoformat(),
        "end_date": end_date.isoformat(),
        "rows_loaded": rows_loaded,
        "files_created": files_created,
        "status": "SUCCESS",
        "duration_seconds": round(duration_seconds,2),
        "last_key": int(last_key)
      }

    try :
        json.dumps(new_record)
    
        history.append(new_record)

        with open("load_history.json", "w") as f:
            json.dump(history, f, indent=4)

    except:
        print("Error")
    
    print(new_record)

    return

In [7]:
#this function is to generate folder name in s3 bucket 
#so basically we will have separate folder for upload on that day 
#this we we can clearly see which data was uploaded on what day
def generate_folder_name(timestamp) :
    ts= timestamp.strftime("%Y%m%d_%H%M%S")
    return f"ingest_date_{ts}"

#### 1.4 S3 Upload Approach

- A unique folder is created for every pipeline run using the current timestamp.
- The folder name follows the format: `ingest_date_YYYYMMDD_HHMMSS`.
- This makes it easy to identify which data was uploaded during a particular ingestion run.
- The raw DataFrame is converted to Parquet format in memory using `BytesIO`, avoiding temporary files on disk.
- The Parquet file is uploaded to Amazon S3 under the path:
  ```
  <base_prefix>/<ingest_date_timestamp>/<file_name>.parquet
  ```
- The function returns the S3 URI of the uploaded file for tracking and downstream processing.

#### Example S3 Path

```text
s3://<bucket-name>/<base_prefix>/ingest_date_20260702_103015/raw_data.parquet
```

In [8]:
def upload_df_to_s3(df, bucket_name, base_prefix, folder_name, file_name):
    
    buffer = BytesIO()

    #dataframe of raw_data
    #converts raw data to parquet and stores it in ram instead of Disk
    df.to_parquet(
        buffer,
        engine="pyarrow",
        index=False
    )
    
    #to bring cursor on start
    buffer.seek(0)

    #to create file path = object key
    key = f"{base_prefix}/{folder_name}/{file_name}.parquet"

    
    s3.upload_fileobj(
        buffer,
        bucket_name,
        key
    )

    return f"s3://{bucket_name}/{key}"

### 2. Data Extraction Strategy
During initial testing, requesting the entire 90-day dataset in a single API call proved unreliable. The web server struggled to process large requests, resulting in increased response times and a higher risk of request timeouts.

To overcome this bottleneck, the ingestion process was redesigned using a combination of time-based chunking and key-based pagination.

- The 90-day extraction window is divided into 9 independent 10-day chunks, reducing the amount of data requested in a single query.
- Each chunk is retrieved in 20,000-record batches, ensuring consistent request sizes.
- Pagination is implemented using the unique_key field instead of offset-based pagination.
- Records are fetched in ascending order of unique_key, providing deterministic ordering and preventing duplicate or skipped records across consecutive requests.

In [9]:
#connect with website API
client = Socrata("data.cityofnewyork.us", None,timeout = 60)

#cutoff = (datetime.now() - timedelta(days=10)).strftime("%Y-%m-%dT%H:%M:%S")
#current date - 90 days is cutoff

batch_size=20000
all_dfs = []

#this way we will create a folder name for upload of that day
ingest_time = datetime.now(ZoneInfo("Asia/Kolkata"))
folder_name = generate_folder_name(ingest_time)

#start_date is basically the day from where we are tracking the data
start_date = (ingest_time - timedelta(days=90))

#start_time is to check overall duration of the process
start_time = time.time()

#to keep a check of total entries & tables uploaded in s3
files_created = 0 
rows_loaded = 0

s3 = boto3.client('s3')

for i in range(9):

    #fetching data in set of 10 days because of bottleneck at Web Server
    chunk_start = start_date + timedelta(days=i*10)
    chunk_end   = chunk_start + timedelta(days=10)

    print(f"Loading Batch :{i} of Data -> From {(str(chunk_start))} to {str(chunk_end)}")
    last_key = 0
    
    while True:               
        results = client.get(
            "erm2-nwe9",
            where = (f"""created_date >= '{chunk_start.strftime("%Y-%m-%dT%H:%M:%S")}' 
            AND created_date < '{chunk_end.strftime("%Y-%m-%dT%H:%M:%S")}'
            AND unique_key > '{last_key}' """ ),
            order = "unique_key ASC",
            limit = batch_size
        )
    
        if not results:
            break

        #converts json results to dataframe 
        df_batch = pd.DataFrame(results)
        all_dfs.append(df_batch)

        last_key = df_batch["unique_key"].astype(int).max()
        load_size = len(df_batch)

        files_created += 1
        rows_loaded += load_size

        try:
            s3_path = upload_df_to_s3(
            df_batch,
            bucket_name='nyi-data-analytics',
            base_prefix='bronze/full_load',
            folder_name=folder_name,
            file_name=f"part_{files_created:03d}"
            )

            s3_upload_status = f"Success: {s3_path}"

        except Exception as error:
            s3_upload_status = f"Failed: {error}"

        print("\t",load_size,"Loaded , Last Key : ", last_key , "Upload Data on S3: ", s3_upload_status)
        
    #break

start_time2= time.time()
dfs = pd.concat(all_dfs, ignore_index=True)
end_time2= time.time()
end_time = time.time()
print("Overall Time",end_time - start_time)
print("Concat Time",end_time2 - start_time2)
#print(df_batch.shape)
print(len(all_dfs))
print(dfs.shape)

#Metadata for logs
print("Files Created : ", files_created) 
run_time = datetime.fromtimestamp(start_time) 
load_type = "FULL"
end_date = start_date + timedelta(days=90)
duration_seconds = end_time - start_time

Loading Batch :0 of Data -> From 2026-04-03 09:10:30.404198+05:30 to 2026-04-13 09:10:30.404198+05:30
	 20000 Loaded , Last Key :  68570282 Upload Data on S3:  Success: s3://nyi-data-analytics/bronze/full_load/ingest_date_20260702_091030/part_001.parquet
	 20000 Loaded , Last Key :  68591464 Upload Data on S3:  Success: s3://nyi-data-analytics/bronze/full_load/ingest_date_20260702_091030/part_002.parquet
	 20000 Loaded , Last Key :  68612792 Upload Data on S3:  Success: s3://nyi-data-analytics/bronze/full_load/ingest_date_20260702_091030/part_003.parquet
	 20000 Loaded , Last Key :  68633858 Upload Data on S3:  Success: s3://nyi-data-analytics/bronze/full_load/ingest_date_20260702_091030/part_004.parquet
	 15529 Loaded , Last Key :  68745181 Upload Data on S3:  Success: s3://nyi-data-analytics/bronze/full_load/ingest_date_20260702_091030/part_005.parquet
Loading Batch :1 of Data -> From 2026-04-13 09:10:30.404198+05:30 to 2026-04-23 09:10:30.404198+05:30
	 20000 Loaded , Last Key :  68

In [10]:
#update metadata 
update_meta_data(end_date,last_key)

LAST UPLOAD METADATA : 
{'last_created_date': '2026-07-01T10:30:44.993371+05:30', 'last_unique_key': 68745181, 'last_ingest_time': '2026-07-01T05:01:22.841660'}
NEW UPLOAD METADATA : 
{'last_created_date': '2026-07-02T09:10:30.404198+05:30', 'last_unique_key': 69550986, 'last_ingest_time': '2026-07-02T03:46:42.487073'}


In [11]:
#update log history of uploads
update_log_history(run_time,start_date,end_date,rows_loaded,files_created,duration_seconds,last_key)

{'run_time': '2026-07-01T05:00:44.993520', 'load_type': 'FULL', 'start_date': '2026-04-02T10:30:44.993371+05:30', 'end_date': '2026-07-01T10:30:44.993371+05:30', 'rows_loaded': 95586, 'files_created': 5, 'status': 'SUCCESS', 'duration_seconds': 37.82, 'last_key': 68745181}
{'run_time': '2026-07-02T03:40:30.404415', 'load_type': 'FULL', 'start_date': '2026-04-03T09:10:30.404198+05:30', 'end_date': '2026-07-02T09:10:30.404198+05:30', 'rows_loaded': 946320, 'files_created': 52, 'status': 'SUCCESS', 'duration_seconds': 372.07, 'last_key': 69550986}
